In [2]:
from pathlib import Path
import os
import pandas as pd
import cdsapi


ROOT = Path("/home/scratch/ocarlisle/era5_483")   # <-- change to your scratch path
CSV_PATH = Path("/home/z2005203/483/erodays.csv")  # <-- your dates CSV
DATE_COL = "date"

AREA = [53, -125, 20, -65]  # [N, W, S, E]
TIME = ["09:00"]

PL_DATASET = "reanalysis-era5-pressure-levels"
SL_DATASET = "reanalysis-era5-single-levels"

PL_DIR = ROOT / "pressure_levels"
SL_DIR = ROOT / "single_levels"
PL_DIR.mkdir(parents=True, exist_ok=True)
SL_DIR.mkdir(parents=True, exist_ok=True)

# I put in some of the vars if you think its useless remove i did this like a month ago
pl_request_base = {
    "product_type": ["reanalysis"],
    "variable": ["geopotential"],
    "pressure_level": ["500", "850"],
    "area": AREA,
}

sl_request_base = {
    "product_type": ["reanalysis"],
    "variable": [
        "total_column_water_vapour",
        "convective_available_potential_energy",
        "mean_sea_level_pressure",
        "total_precipitation",
    ],
    "area": AREA
}


def retrieve_month_chunk(client, dataset_short_name, base_request, year, month, days, target_path: Path):
    req = dict(base_request)
    req["year"] = [f"{year:04d}"]
    req["month"] = [f"{month:02d}"]
    req["day"] = [f"{d:02d}" for d in sorted(set(days))]
    req["time"] = TIME

    # CDS has used both 'format' and 'data_format' depending on UI/version.
    # We'll set both safely; CDS will ignore what it doesn't need.
    req.setdefault("data_format", "netcdf")
    req.setdefault("format", "netcdf")

    if target_path.exists():
        print(f"Skipping (already exists): {target_path}")
        return

    print(f"Requesting {dataset_short_name} {year}-{month:02d} ({len(req['day'])} days) -> {target_path}")
    client.retrieve(dataset_short_name, req, str(target_path))


def main():
    # cdsapi reads ~/.cdsapirc by default
    client = cdsapi.Client()

    df = pd.read_csv(CSV_PATH)
    if DATE_COL not in df.columns:
        raise ValueError(f"CSV must contain column '{DATE_COL}'. Found: {list(df.columns)}")

    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    df = df.dropna(subset=[DATE_COL]).drop_duplicates(subset=[DATE_COL]).sort_values(DATE_COL)

    # Use date only for grouping; time is always 09:00 requested above
    df["year"] = df[DATE_COL].dt.year
    df["month"] = df[DATE_COL].dt.month
    df["day"] = df[DATE_COL].dt.day

    for (year, month), g in df.groupby(["year", "month"]):
        days = g["day"].tolist()

        pl_out = PL_DIR / f"era5_pl_{year:04d}{month:02d}_09z_real.nc"
        sl_out = SL_DIR / f"era5_sl_{year:04d}{month:02d}_09z_real.nc"

        retrieve_month_chunk(client, PL_DATASET, pl_request_base, year, month, days, pl_out)
        retrieve_month_chunk(client, SL_DATASET, sl_request_base, year, month, days, sl_out)


if __name__ == "__main__":
    main()

Requesting reanalysis-era5-pressure-levels 2020-01 (6 days) -> /home/scratch/ocarlisle/era5_483/pressure_levels/era5_pl_202001_09z_real.nc


2026-03-27 22:21:13,990 INFO Request ID is e187acb8-6cf0-46c5-aed1-e7fe56239acf
2026-03-27 22:21:14,144 INFO status has been updated to accepted
2026-03-27 22:21:35,804 INFO status has been updated to running
2026-03-27 22:21:47,352 INFO status has been updated to successful


900f4a2d1f62e468433044e2f44fde68.nc:   0%|          | 0.00/576k [00:00<?, ?B/s]

Requesting reanalysis-era5-single-levels 2020-01 (6 days) -> /home/scratch/ocarlisle/era5_483/single_levels/era5_sl_202001_09z_real.nc


2026-03-27 22:21:50,495 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-03-27 22:21:50,497 INFO Request ID is d4381670-6053-4dd9-98ba-f86cbd86bb36
2026-03-27 22:21:50,630 INFO status has been updated to accepted
2026-03-27 22:22:04,516 INFO status has been updated to running
2026-03-27 22:22:23,790 INFO status has been updated to successful


...

2026-03-27 23:37:21,679 INFO Request ID is e6159c49-9f5d-4487-a65b-1de57e8adb8a
2026-03-27 23:37:21,812 INFO status has been updated to accepted
2026-03-27 23:37:43,629 INFO status has been updated to successful


95ee9b40054bf6a51f3d1ac21eca2ffd.nc:   0%|          | 0.00/123k [00:00<?, ?B/s]

Requesting reanalysis-era5-single-levels 2025-12 (1 days) -> /home/scratch/ocarlisle/era5_483/single_levels/era5_sl_202512_09z_real.nc


2026-03-27 23:37:46,433 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-03-27 23:37:46,436 INFO Request ID is fbc43176-3537-4cba-a997-93afa0b9b293
2026-03-27 23:37:46,580 INFO status has been updated to accepted
2026-03-27 23:38:08,217 INFO status has been updated to successful


f30827eb14108cc4a87add3dc986fc8f.zip:   0%|          | 0.00/231k [00:00<?, ?B/s]